# NC-SSM-Large + SS: TASLP Final Experiments
**10.2K params | 4.1x MAC advantage | SS+Bypass v4 optimized**

---

## Execution Order

| Step | Description | Time |
|------|-------------|------|
| **0** | Setup (GPU + Drive) | 30s |
| **1** | Clone + Data + Checkpoints | ~3min |
| **2** | Train NC-SSM-Large (10.2K) | ~10min |
| **3** | Full 7-SNR Noise Evaluation | ~15min |
| **4** | SS+Bypass v4 Evaluation | ~15min |
| **5** | SS+Bypass Optimization Sweep | ~30min |
| **6** | SM-SSM + SAGN Training (ablation) | ~20min |
| **7** | Full 6-Model Comparison | ~20min |
| **8** | Sigma Gate Analysis | ~5min |

## Step 0: Setup

In [ ]:
# GPU check
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Mount Drive (for checkpoint persistence)
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Clone + Data + Checkpoints

In [ ]:
%%time
import os, shutil, torch

# Clone latest code from NC-SSM-TASLP repo
!rm -rf /content/NC-SSM-TASLP
!git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP.git /content/NC-SSM-TASLP
os.chdir('/content/NC-SSM-TASLP')

# Download GSC V2
!python train_colab.py --models NanoMamba-Tiny --epochs 0 --data_dir ./data 2>/dev/null || true
if not os.path.exists('./data/speech_commands_v0.02'):
    !mkdir -p data && cd data && \
     wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz && \
     mkdir -p speech_commands_v0.02 && \
     tar -xzf speech_commands_v0.02.tar.gz -C speech_commands_v0.02 && \
     rm speech_commands_v0.02.tar.gz

# Restore checkpoints from Drive (if available)
DRIVE_CKPT = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
LOCAL_CKPT = './checkpoints_full'
if os.path.exists(DRIVE_CKPT) and not os.path.exists(LOCAL_CKPT):
    shutil.copytree(DRIVE_CKPT, LOCAL_CKPT)
    print(f'Restored checkpoints from Drive')
os.makedirs(LOCAL_CKPT, exist_ok=True)

# Show available checkpoints
for d in sorted(os.listdir(LOCAL_CKPT)):
    best = os.path.join(LOCAL_CKPT, d, 'best.pt')
    if os.path.exists(best):
        ckpt = torch.load(best, map_location='cpu')
        acc = ckpt.get('val_acc', 'N/A')
        print(f'  {d}: val_acc={acc}')

## Step 2: Train NC-SSM-Large (10.2K params)
- `d_model=24, d_state=8`: 10,201 params
- **1.15M MACs** (4.1x advantage over BC-ResNet-1)
- Cortex-M7: **2.4ms latency**

In [ ]:
# Train NC-SSM-Large (same recipe as NC-SSM)
!python train_colab.py \
    --models NanoMamba-NC-Large \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --spec_augment --use_ema \
    --calibrate

# Save to Drive
import shutil
DRIVE_CKPT = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
os.makedirs(DRIVE_CKPT, exist_ok=True)
shutil.copytree('./checkpoints_full', DRIVE_CKPT, dirs_exist_ok=True)
print('Checkpoints saved to Drive')

## Step 3: Full 7-SNR Noise Evaluation
NC-SSM-Large vs NC-SSM vs BC-ResNet-1

In [ ]:
# Full noise evaluation: 5 noise x 7 SNR = 35 conditions
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

## Step 4: SS+Bypass v4 Evaluation
**sf_range=2.0 + SF-weighted subtraction**

In [ ]:
# SS+Bypass v4 (current optimized defaults)
# --use_enhancer: enable spectral subtraction front-end
# --enhancer_bypass: SNR-adaptive bypass gate (preserve clean performance)
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

## Step 5: SS+Bypass Optimization Sweep
**v4 파라미터가 최적인지 확인**: sf_range, bypass_threshold, bypass_scale

In [ ]:
# ============================================================
# SS+Bypass Parameter Sweep: Is v4 (sf_range=2.0) optimal?
# Test 4 configurations and compare at -15dB and 0dB
# ============================================================

configs = [
    # (name, sf_range, threshold, scale)
    ('v4-current',  2.0, -2.0, 3.0),  # Current default
    ('v4-tight',    1.0, -1.0, 4.0),  # Tighter bypass: more aggressive at 0dB
    ('v4-wide',     3.0, -3.0, 2.0),  # Wider range: more SS at moderate SNR
    ('v4-noadapt',  0.0, -2.0, 3.0),  # No SF adaptation: same threshold all noises
]

print('='*70)
print('SS+BYPASS v4 PARAMETER SWEEP')
print('='*70)

for name, sf_range, threshold, scale in configs:
    print(f'\n{"="*70}')
    print(f'Config: {name} (sf_range={sf_range}, threshold={threshold}, scale={scale})')
    print(f'{"="*70}')
    !python train_colab.py \
        --models NanoMamba-NC-Large \
        --eval_only \
        --use_enhancer --enhancer_bypass \
        --ss_version v2 --bypass_version v2 \
        --bypass_threshold {threshold} --bypass_scale {scale} --sf_range {sf_range} \
        --data_dir ./data \
        --checkpoint_dir ./checkpoints_full \
        --noise_types factory,white,babble,street,pink \
        --snr_range=-15,0 \
        --calibrate --calibrate_continuous

print('\n' + '='*70)
print('SWEEP COMPLETE: Compare -15dB improvement vs 0dB degradation')
print('Optimal config should maximize -15dB gain while minimizing 0dB loss')
print('='*70)

## Step 6: SM-SSM + SAGN Training (TASLP ablation baselines)
논문 스토리: NM-Matched → SM-SSM → NC-SSM 진화

In [ ]:
# Train SM-SSM (intermediate step: +38 params for sigma gate)
!python train_colab.py \
    --models NanoMamba-Matched-SM \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --spec_augment --use_ema \
    --calibrate

# Train SAGN (CNN+LSG baseline: proves LSG alone on CNN < NC-SSM)
!python train_colab.py \
    --models SAGN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --spec_augment --use_ema \
    --calibrate

# Save to Drive
DRIVE_CKPT = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
shutil.copytree('./checkpoints_full', DRIVE_CKPT, dirs_exist_ok=True)
print('All checkpoints saved to Drive')

## Step 7: Full 6-Model TASLP Comparison
**All models × 5 noise × 7 SNR = complete Table V**

In [ ]:
# TASLP Table V: Complete 6-model noise evaluation
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,SAGN,BC-ResNet-1,NanoMamba-Matched-DualPCEN-v2-SSMv2,DS-CNN-S \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# TASLP Table VII: SS+Bypass for all models
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,NanoMamba-Matched-DualPCEN-v2-SSMv2,BC-ResNet-1,DS-CNN-S,SAGN \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*60)
print('TASLP FULL COMPARISON COMPLETE')
print('='*60)

## Step 8: Sigma Gate Analysis (Fig. 3 data)
Per-sub-band selectivity visualization data

In [ ]:
# Sigma gate per-sub-band analysis for NC-SSM-Large
import torch, sys, os
sys.path.insert(0, '/content/NC-SSM-TASLP')
from train_colab import (
    SpeechCommandsDataset, analyze_selectivity_gates,
    generate_noise_signal, mix_audio_at_snr
)
from nanomamba import create_nanomamba_nc_large

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load NC-SSM-Large checkpoint
model = create_nanomamba_nc_large(n_classes=12).to(device)
ckpt_path = './checkpoints_full/NanoMamba-NC-Large/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded NC-SSM-Large: val_acc={ckpt.get("val_acc", "N/A")}')
else:
    print(f'Checkpoint not found: {ckpt_path}')
    print('Run Step 2 first!')

model.eval()

# Load dataset for analysis
val_dataset = SpeechCommandsDataset('./data/speech_commands_v0.02', split='validation')

# Analyze sigma gates
analyze_selectivity_gates(model, val_dataset, device=device)